In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [3]:
df = pd.read_csv("../data/raw/overtaking_data.csv")

In [4]:
X = df.drop(columns=["decision"])
y = df["decision"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, class_weight="balanced", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric="mlogloss", random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, class_weight="balanced", random_state=42, verbose=-1, n_jobs=-1),
    "CatBoost": CatBoostClassifier(iterations=150, auto_class_weights="Balanced", verbose=0, random_seed=42)
}

In [8]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    macro_f1 = f1_score(y_test, pred, average="macro")
    results[name] = macro_f1
    print(f"{name:20s} | Acc: {accuracy_score(y_test, pred):.4f} | Macro F1: {macro_f1:.4f}")

best_name = max(results, key=results.get)
print(f"\nBest Model: {best_name} (Macro F1: {results[best_name]:.4f})")

Logistic Regression  | Acc: 0.9532 | Macro F1: 0.9315
KNN                  | Acc: 0.9772 | Macro F1: 0.9643
Decision Tree        | Acc: 0.9799 | Macro F1: 0.9685
Random Forest        | Acc: 0.9816 | Macro F1: 0.9712
Gradient Boosting    | Acc: 0.9836 | Macro F1: 0.9741
XGBoost              | Acc: 0.9810 | Macro F1: 0.9700
LightGBM             | Acc: 0.9820 | Macro F1: 0.9720
CatBoost             | Acc: 0.9815 | Macro F1: 0.9712

Best Model: Gradient Boosting (Macro F1: 0.9741)


In [9]:
from sklearn.metrics import classification_report, confusion_matrix

best_model = models[best_name]
pred = best_model.predict(X_test)

print(f"=== {best_name} Detailed Report ===")
print(classification_report(y_test, pred, target_names=["KEEP (0)", "LEFT (1)", "RIGHT (2)"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

=== Gradient Boosting Detailed Report ===
              precision    recall  f1-score   support

    KEEP (0)       0.99      0.99      0.99      7401
    LEFT (1)       0.96      0.95      0.95      1286
   RIGHT (2)       0.98      0.98      0.98      1313

    accuracy                           0.98     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.98      0.98      0.98     10000

Confusion Matrix:
 [[7326   49   26]
 [  58 1222    6]
 [  21    4 1288]]
